<a href="https://colab.research.google.com/github/book150243/Homework_DADS6003/blob/main/HW_DADS6003_6810422005.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss

In [2]:
#upload text file
leukemia_df = pd.read_csv("leukemia_remission.txt",sep = "\t")

#find correlation
corr_matrix = leukemia_df.corr()
remiss_corr = corr_matrix["REMISS"].sort_values(ascending = False)
print(remiss_corr)

plt.figure(figsize=(6,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix of Leukemia dataset")
plt.show()

#define x and y
x = leukemia_df.drop(["REMISS", "TEMP", "SMEAR"], axis = 1) #or x = leukemia_df.iloc[:, 1:]
y = leukemia_df["REMISS"]

FileNotFoundError: [Errno 2] No such file or directory: 'leukemia_remission.txt'

# Logistic Regression

In [ ]:
#Split data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 24)

#Train: use logistics regression to fit x_train and y_train and get model
logreg = LogisticRegression()
logreg.fit(x_train,y_train)

#Test
y_pred = logreg.predict(x_test)
y_pred_proba = logreg.predict_proba(x_test)

#evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"theta0 : {logreg.intercept_[0]:.4f}")

for i in range(len(logreg.coef_[0])):
    print(f"theta{i+1} : {logreg.coef_[0][i]:.4f}")

proba_df = pd.DataFrame(y_pred_proba, columns=['Probability_Class_0', 'Probability_Class_1'])


compare_y = pd.DataFrame({
    'y_test':y_test,
    'y_pred':y_pred
})
display(proba_df)
display(compare_y)

# BGD

In [ ]:
import numpy as np

#cal sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

#cal cost function
def cost_function(x, y, theta):
    N = x.shape[0]
    h = sigmoid(np.matmul(x, theta))
    cost = - (1/N) * (np.dot(y.T, np.log(h)) + np.dot((1 - y.T), np.log(1 - h)))
    return cost[0][0]

#cal Gradient Descent
def gradient_descent(theta, eta, x, y):
    N = x.shape[0]
    h = sigmoid(np.matmul(x, theta))
    error = h - y
    grad = (1/N) * np.matmul(x.T, error)
    theta = theta - (eta * grad)
    return theta

x_t = np.c_[np.ones((x.shape[0], 1)), x]
y_t = y.values.reshape(-1, 1)

#learning rate
eta = 0.1
iterations = 500000
theta = np.random.randn(x_t.shape[1], 1)

cost_n = cost_function(x_t, y_t, theta)

for i in range(iterations):
    theta = gradient_descent(theta, eta, x_t, y_t)
    cost_n_plus_1 = cost_function(x_t, y_t, theta)

    if cost_n_plus_1 > cost_n:
        print(f"Stopping at iteration {i}: Cost increased from {cost_n:.6f} to {cost_n_plus_1:.6f}")
        break

    if i % 100000 == 0:
        print(f"Iteration {i}: NLLL = {cost_function(x_t, y_t, theta):.6f}")

    cost_n = cost_n_plus_1

#predict
h = sigmoid(np.matmul(x_t, theta))
y_hat = (h > 0.5).astype(int)

print("Actual Y:", y_t.flatten())
print("Predicted Y:", y_hat.flatten())

acc = accuracy_score(y_t, y_hat)
print(f"Accuracy score: {acc*100:.2f}%")